<a href="https://colab.research.google.com/github/0MSA0/supreme-robot/blob/main/data_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Data-Loader

DataSet Class:

In [18]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class SupremeDataset(Dataset):
  def __init__(self, input_text: str, tokenizer, max_length, stride) -> None:
    self.input_ids = []
    self.target_ids = []

    encoded_words = tokenizer.encode(input_text)

    # Use a sliding window to chunk the input text into overlapping sequences of max_length
    for i in range(0, len(encoded_words) - max_length, stride):
      input_chunk = encoded_words[i:i + max_length]
      target_chunk = encoded_words[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]


DataLoader:

In [19]:
def create_supreme_dataloader(txt, tokenizer, batch_size=4, max_length=256,
                              stride=128, shuffle=True, drop_last=True,
                              num_workers=0):

    # Create dataset
    dataset = SupremeDataset(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Function for creating the embeddings:

In [20]:
def create_embeddings(raw_text, tokenizer, output_dim=256, context_length=1024,
                      batch_size=8, max_length=4, stride=4, shuffle=True,
                      drop_last=True, num_workers=0):
  vocab_size = tokenizer.n_vocab

  token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
  pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

  dataloader = create_supreme_dataloader(
      raw_text,
      tokenizer,
      batch_size=batch_size,
      max_length=max_length,
      stride=max_length
  )

  input_embeddings = []
  target_embeddings = []
  for batch in dataloader:
      input_token_ids, target_token_ids = batch
      cur_pos_embedding = pos_embedding_layer(torch.arange(max_length))

      cur_input_token_embedding = token_embedding_layer(input_token_ids)
      cur_input_embedding = cur_input_token_embedding + cur_pos_embedding

      cur_target_token_embedding = token_embedding_layer(target_token_ids)
      cur_target_embedding = cur_target_token_embedding + cur_pos_embedding

      input_embeddings.append(cur_input_embedding)
      target_embeddings.append(cur_target_embedding)

  return input_embeddings, target_embeddings


Open sample text:

In [21]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

Create input-embeddings and target-embeddings for sample text:

In [22]:
tokenizer = tiktoken.get_encoding("o200k_base")
input_embeddings, target_embeddings = create_embeddings(raw_text, tokenizer)
print(input_embeddings[0].shape)
print(target_embeddings[0].shape)

torch.Size([8, 4, 256])
torch.Size([8, 4, 256])
